In [2]:
!pip -q install pyspark

In [7]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload() # Browse and select all six CSVs
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()
def load_csv(filename):
  return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")

Saving maintenance_tickets_1600.csv to maintenance_tickets_1600 (1).csv
Saving meter_master.csv to meter_master (1).csv
Saving meter_readings_90000.csv to meter_readings_90000 (1).csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000 (1).csv
Saving substation_master.csv to substation_master (1).csv
Saving tariff_master.csv to tariff_master (1).csv


In [8]:
print("Main readings:", readings_df.count())
print("Extra readings:", readings_extra_df.count())
print("Meter master:", meter_master_df.count())
print("Substation master:", substation_master_df.count())
print("Tariff master:", tariff_master_df.count())
print("Maintenance tickets:", maintenance_df.count())

Main readings: 90000
Extra readings: 12000
Meter master: 150
Substation master: 12
Tariff master: 5
Maintenance tickets: 1600


**Tasks 1-13**

In [9]:
readings_df.printSchema()

print("Total readings:", readings_df.count())

readings_df.show(10, truncate=False)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Total readings: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0000001 |M0082   |20

In [12]:
selected_df = readings_df.select(
    "reading_id",
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "reading_status"
)

renamed_df = readings_df.withColumnRenamed(
    "source_type",
    "collection_mode"
)

selected_df.show(10, truncate=False)
renamed_df.show(10, truncate=False)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
|RD0000001 |M0082   |2026-04-01 02:00:00|22.53     |Valid         |
|RD0000002 |M0020   |2026-01-31 06:00:00|2.97      |Valid         |
|RD0000003 |M0054   |2026-04-11 10:15:00|8.61      |Estimated     |
|RD0000004 |M0111   |2026-03-14 00:00:00|1.87      |Valid         |
|RD0000005 |M0135   |2026-05-27 01:45:00|4.39      |Valid         |
|RD0000006 |M0139   |2026-02-20 02:15:00|0.36      |Valid         |
|RD0000007 |M0142   |2026-02-01 05:30:00|79.47     |Valid         |
|RD0000008 |M0126   |2026-03-01 20:45:00|28.35     |Rejected      |
|RD0000009 |M0005   |2026-05-03 10:30:00|1.96      |Estimated     |
|RD0000010 |M0052   |2026-04-10 05:45:00|22.52     |Valid         |
+----------+--------+-------------------+----------+--------------+
only showing top 10 rows
+----------+--------+--

In [13]:
valid_readings_df = readings_df.filter(
    (F.col("reading_status") == "Valid") &
    (F.col("energy_kwh") > 20)
)

sorted_readings_df = valid_readings_df.orderBy(
    F.col("energy_kwh").desc()
)

sorted_readings_df.show(20, truncate=False)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0041703 |M0101   |2026-03-06 22:30:00|110.0     |213.1    |0.822       |0             |Valid         |SmartMeter |31.1                 |OP010      |
|RD0077494 |M0014   |2026-05-05 17:30:00|109.99    |NULL     |0.771       |0             |Valid         |Manual     |39.0                 |OP031      |
|RD0034412 |M0026   |2026-02-24 21:15:00|109.96    |231.6    |0.964       |0             |Valid         |SmartMeter |25.3                 |OP017      |
|RD0039063 |M0101   |2026-02-01 00:45:00|109.96    |231.6    |0.835       |0            

In [14]:
energy_band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
)

energy_band_df.select(
    "reading_id",
    "energy_kwh",
    "energy_band"
).show(20)

+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
| RD0000001|     22.53|     Medium|
| RD0000002|      2.97|        Low|
| RD0000003|      8.61|     Medium|
| RD0000004|      1.87|        Low|
| RD0000005|      4.39|        Low|
| RD0000006|      0.36|        Low|
| RD0000007|     79.47|       High|
| RD0000008|     28.35|       High|
| RD0000009|      1.96|        Low|
| RD0000010|     22.52|     Medium|
| RD0000011|     10.51|     Medium|
| RD0000012|      1.49|        Low|
| RD0000013|     15.75|     Medium|
| RD0000014|     15.96|     Medium|
| RD0000015|      6.06|     Medium|
| RD0000016|     25.82|       High|
| RD0000017|      6.23|     Medium|
| RD0000018|      3.66|        Low|
| RD0000019|     104.8|       High|
| RD0000020|      1.35|        Low|
+----------+----------+-----------+
only showing top 20 rows


In [15]:
null_counts_df = readings_df.select(
    F.sum(F.col("energy_kwh").isNull().cast("int")).alias("energy_kwh_nulls"),
    F.sum(F.col("voltage_v").isNull().cast("int")).alias("voltage_v_nulls"),
    F.sum(F.col("power_factor").isNull().cast("int")).alias("power_factor_nulls"),
    F.sum(F.col("operator_id").isNull().cast("int")).alias("operator_id_nulls")
)

null_counts_df.show()

+----------------+---------------+------------------+-----------------+
|energy_kwh_nulls|voltage_v_nulls|power_factor_nulls|operator_id_nulls|
+----------------+---------------+------------------+-----------------+
|            1620|           2070|              1440|             5400|
+----------------+---------------+------------------+-----------------+



In [16]:
filled_df = readings_df.fillna({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})

filled_df.select(
    "operator_id",
    "power_factor"
).show(20)

print("Original row count:", readings_df.count())
print("Filled DataFrame row count:", filled_df.count())

+-----------+------------+
|operator_id|power_factor|
+-----------+------------+
|      OP029|       0.939|
|      OP025|       0.845|
|      OP007|       0.879|
| UNASSIGNED|       0.831|
|      OP019|       0.966|
|      OP021|       0.788|
|      OP016|       0.986|
|      OP025|       0.818|
|      OP010|       0.817|
|      OP028|       0.804|
|      OP030|         0.9|
|      OP027|        0.81|
|      OP005|       0.807|
|      OP019|       0.884|
|      OP036|       0.919|
|      OP035|       0.895|
|      OP003|       0.762|
|      OP021|       0.914|
|      OP014|       0.953|
|      OP020|       0.771|
+-----------+------------+
only showing top 20 rows
Original row count: 90000
Filled DataFrame row count: 90000


In [17]:
cleaned_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() &
    F.col("voltage_v").isNotNull()
)

original_count = readings_df.count()
cleaned_count = cleaned_df.count()

print("Original rows:", original_count)
print("Rows after cleaning:", cleaned_count)
print("Rows removed:", original_count - cleaned_count)

Original rows: 90000
Rows after cleaning: 86349
Rows removed: 3651


In [18]:
meter_join_df = readings_df.join(
    meter_master_df,
    on="meter_id",
    how="inner"
)

site_energy_df = meter_join_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "site_type"
).agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)

site_energy_df.show(truncate=False)

+---------------+-----------------+
|site_type      |total_energy_kwh |
+---------------+-----------------+
|Public Facility|138870.41        |
|Residential    |100863.2900000004|
|Industrial     |710317.4100000021|
|Commercial     |451525.1799999996|
+---------------+-----------------+



In [19]:
substation_join_df = meter_join_df.join(
    substation_master_df,
    on="substation_id",
    how="left"
)

region_city_energy_df = substation_join_df.filter(
    F.col("reading_status") == "Valid"
).groupBy(
    "region",
    "city"
).agg(
    F.sum("energy_kwh").alias("total_energy_kwh")
)

region_city_energy_df.show(truncate=False)

+-------+-------------+------------------+
|region |city         |total_energy_kwh  |
+-------+-------------+------------------+
|Central|Indore       |176503.91000000012|
|West   |Mumbai       |52116.20999999999 |
|South  |Hyderabad    |51937.789999999884|
|South  |Visakhapatnam|205015.82999999932|
|West   |Ahmedabad    |138627.7400000004 |
|North  |Lucknow      |142610.3999999998 |
|South  |Bengaluru    |73787.59999999998 |
|North  |Jaipur       |47863.15999999996 |
|South  |Kochi        |204272.4600000002 |
|North  |Delhi        |94234.89000000014 |
|South  |Chennai      |51829.05999999996 |
|West   |Pune         |162777.23999999996|
+-------+-------------+------------------+



In [ ]:
tariff_join_df = meter_join_df.join(
    tariff_master_df,
    on="tariff_id",
    how="left"
)

energy_charges_df = tariff_join_df.filter(
    (F.col("reading_status") != "Rejected") &
    F.col("energy_kwh").isNotNull()
).withColumn(
    "energy_charge_inr",
    F.col("energy_kwh") * F.col("rate_per_kwh_inr")
)

energy_charges_df.select(
    "reading_id",
    "meter_id",
    "energy_kwh",
    "rate_per_kwh_inr",
    "energy_charge_inr"
).show(20, truncate=False)

In [20]:
maintenance_join_df = maintenance_df.join(
    meter_master_df,
    on="meter_id",
    how="left"
)

maintenance_summary_df = maintenance_join_df.groupBy(
    "site_type",
    "severity"
).agg(
    F.count("*").alias("ticket_count"),
    F.avg("maintenance_cost_inr").alias("average_maintenance_cost_inr")
)

maintenance_summary_df.show(truncate=False)

+---------------+--------+------------+----------------------------+
|site_type      |severity|ticket_count|average_maintenance_cost_inr|
+---------------+--------+------------+----------------------------+
|Residential    |Critical|41          |11875.782702702705          |
|Industrial     |Medium  |106         |12043.898526315788          |
|Commercial     |Critical|37          |13424.726216216219          |
|Commercial     |Low     |166         |12223.517692307689          |
|Public Facility|High    |48          |12905.279347826086          |
|Residential    |Low     |194         |12531.90201117318           |
|Public Facility|Medium  |97          |13050.628421052634          |
|Industrial     |Critical|17          |12296.198000000002          |
|Public Facility|Critical|15          |8986.843571428572           |
|Industrial     |High    |63          |12838.869344262299          |
|Residential    |High    |156         |11047.967122302161          |
|Public Facility|Low     |60      

In [23]:
readings_meter_df = readings_df.join(
    meter_master_df,
    on="meter_id",
    how="left"
)

substation_summary_df = readings_meter_df.groupBy(
    "substation_id"
).agg(
    F.avg("voltage_v").alias("average_voltage_v"),
    F.sum("outage_minutes").alias("total_outage_minutes")
)

top_5_substations_df = substation_summary_df.orderBy(
    F.col("total_outage_minutes").desc_nulls_last()
).limit(5)

top_5_substations_df.show(truncate=False)

+-------------+------------------+--------------------+
|substation_id|average_voltage_v |total_outage_minutes|
+-------------+------------------+--------------------+
|SUB10        |230.04010567581344|91224               |
|SUB09        |230.04586272640606|81734               |
|SUB04        |230.0558687466385 |75872               |
|SUB08        |229.88012557077673|75581               |
|SUB03        |230.03094896227708|61662               |
+-------------+------------------+--------------------+



In [24]:
source_status_counts_df = readings_df.groupBy(
    "source_type",
    "reading_status"
).agg(
    F.count("*").alias("reading_count")
).orderBy(
    F.col("reading_count").desc()
)

source_status_counts_df.show(truncate=False)

+-----------+--------------+-------------+
|source_type|reading_status|reading_count|
+-----------+--------------+-------------+
|SmartMeter |Valid         |65521        |
|SmartMeter |Estimated     |10443        |
|Manual     |Valid         |8965         |
|SmartMeter |Rejected      |3202         |
|Manual     |Estimated     |1405         |
|Manual     |Rejected      |464          |
+-----------+--------------+-------------+



**Tasks 14-25**

In [25]:
timestamp_readings_df = readings_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
).withColumn(
    "reading_year", F.year("reading_timestamp")
).withColumn(
    "reading_month", F.month("reading_timestamp")
)

monthly_valid_region_df = (
    timestamp_readings_df
    .filter(F.col("reading_status") == "Valid")
    .join(
        meter_master_df.select("meter_id", "substation_id"),
        on="meter_id",
        how="left"
    )
    .join(
        substation_master_df.select("substation_id", "region"),
        on="substation_id",
        how="left"
    )
    .groupBy("reading_year", "reading_month", "region")
    .agg(
        F.sum("energy_kwh").alias("monthly_valid_energy_kwh")
    )
    .orderBy("reading_year", "reading_month", "region")
)

monthly_valid_region_df.show(20, truncate=False)

+------------+-------------+-------+------------------------+
|reading_year|reading_month|region |monthly_valid_energy_kwh|
+------------+-------------+-------+------------------------+
|2026        |1            |Central|31475.57999999999       |
|2026        |1            |North  |48849.309999999925      |
|2026        |1            |South  |97547.38999999987       |
|2026        |1            |West   |59967.56999999997       |
|2026        |2            |Central|28161.410000000007      |
|2026        |2            |North  |45444.249999999985      |
|2026        |2            |South  |90864.80000000006       |
|2026        |2            |West   |55807.89999999989       |
|2026        |3            |Central|29805.85                |
|2026        |3            |North  |51094.61000000006       |
|2026        |3            |South  |102258.77000000005      |
|2026        |3            |West   |60480.58999999995       |
|2026        |4            |Central|29475.339999999982      |
|2026   

In [26]:
meter_energy_df = (
    readings_df
    .filter(F.col("reading_status") == "Valid")
    .join(
        meter_master_df.select("meter_id", "substation_id"),
        on="meter_id",
        how="left"
    )
    .groupBy("substation_id", "meter_id")
    .agg(
        F.sum("energy_kwh").alias("total_valid_energy_kwh")
    )
)

substation_rank_window = Window.partitionBy(
    "substation_id"
).orderBy(
    F.col("total_valid_energy_kwh").desc()
)

top_three_meters_df = (
    meter_energy_df
    .withColumn(
        "energy_rank",
        F.dense_rank().over(substation_rank_window)
    )
    .filter(F.col("energy_rank") <= 3)
    .orderBy("substation_id", "energy_rank")
)

top_three_meters_df.show(50, truncate=False)

+-------------+--------+----------------------+-----------+
|substation_id|meter_id|total_valid_energy_kwh|energy_rank|
+-------------+--------+----------------------+-----------+
|SUB01        |M0130   |9364.98               |1          |
|SUB01        |M0110   |9259.750000000002     |2          |
|SUB01        |M0068   |8832.580000000005     |3          |
|SUB02        |M0071   |29242.04              |1          |
|SUB02        |M0145   |28641.319999999992    |2          |
|SUB02        |M0133   |7060.140000000003     |3          |
|SUB03        |M0096   |9632.740000000003     |1          |
|SUB03        |M0080   |9525.940000000002     |2          |
|SUB03        |M0073   |9369.789999999995     |3          |
|SUB04        |M0038   |29505.88              |1          |
|SUB04        |M0099   |29216.319999999996    |2          |
|SUB04        |M0098   |28613.460000000014    |3          |
|SUB05        |M0094   |9659.639999999996     |1          |
|SUB05        |M0050   |9570.85999999999

In [27]:
cumulative_window = (
    Window.partitionBy("meter_id")
    .orderBy("reading_timestamp", "reading_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

cumulative_energy_df = (
    readings_df
    .withColumn(
        "reading_timestamp",
        F.to_timestamp("reading_timestamp")
    )
    .filter(F.col("reading_status") == "Valid")
    .withColumn(
        "cumulative_valid_energy_kwh",
        F.sum("energy_kwh").over(cumulative_window)
    )
    .orderBy("meter_id", "reading_timestamp", "reading_id")
)

cumulative_energy_df.select(
    "meter_id",
    "reading_id",
    "reading_timestamp",
    "energy_kwh",
    "cumulative_valid_energy_kwh"
).show(20, truncate=False)

+--------+----------+-------------------+----------+---------------------------+
|meter_id|reading_id|reading_timestamp  |energy_kwh|cumulative_valid_energy_kwh|
+--------+----------+-------------------+----------+---------------------------+
|M0001   |RD0083587 |2026-01-01 01:15:00|22.27     |22.27                      |
|M0001   |RD0086275 |2026-01-01 03:30:00|4.05      |26.32                      |
|M0001   |RD0088204 |2026-01-01 09:45:00|11.28     |37.6                       |
|M0001   |RD0005149 |2026-01-02 04:45:00|11.76     |49.36                      |
|M0001   |RD0035530 |2026-01-02 08:15:00|10.06     |59.42                      |
|M0001   |RD0070104 |2026-01-02 18:30:00|15.83     |75.25                      |
|M0001   |RD0006677 |2026-01-03 00:30:00|17.06     |92.31                      |
|M0001   |RD0021042 |2026-01-03 06:00:00|17.14     |109.45                     |
|M0001   |RD0032205 |2026-01-03 08:30:00|20.46     |129.91                     |
|M0001   |RD0074606 |2026-01

In [28]:
main_count = readings_df.count()
extra_count = readings_extra_df.count()

combined_readings_df = readings_df.unionByName(
    readings_extra_df,
    allowMissingColumns=True
)

combined_count = combined_readings_df.count()

print("Main readings count:", main_count)
print("Extra readings count:", extra_count)
print("Expected combined count:", main_count + extra_count)
print("Actual combined count:", combined_count)

combined_readings_df.show(10, truncate=False)

Main readings count: 90000
Extra readings count: 12000
Expected combined count: 102000
Actual combined count: 102000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0000001 |M0082   |2026-04-01 02:00:00|22.53     |213.1    |0.939       |0             |Valid         |SmartMeter |36.2                 |OP029      |
|RD0000002 |M0020   |2026-01-31 06:00:00|2.97      |227.4    |0.845       |0             |Valid         |SmartMeter |41.2                 |OP025      |
|RD0000003 |M0054   |2026-04-11 10:15:00|8.61      |239.3    |0.879       |0             |Estimated     |SmartMeter |20.9  

In [29]:
duplicate_readings_df = (
    combined_readings_df
    .groupBy("reading_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Repeated reading_id values:")
duplicate_readings_df.show(20, truncate=False)

unique_readings_df = combined_readings_df.dropDuplicates(
    ["reading_id"]
)

print("Rows before deduplication:", combined_readings_df.count())
print("Rows after deduplication:", unique_readings_df.count())
print("Unique reading IDs:", unique_readings_df.select("reading_id").distinct().count())

Repeated reading_id values:
+----------+-----+
|reading_id|count|
+----------+-----+
|RD0025848 |2    |
|RD0045533 |2    |
|RD0064921 |2    |
|RD0056575 |2    |
|RD0058815 |2    |
|RD0022131 |2    |
|RD0029598 |2    |
|RD0045710 |2    |
|RD0030877 |2    |
|RD0002679 |2    |
|RD0011169 |2    |
|RD0028736 |2    |
|RD0030669 |2    |
|RD0061933 |2    |
|RD0015904 |2    |
|RD0008188 |2    |
|RD0037899 |2    |
|RD0000004 |2    |
|RD0025628 |2    |
|RD0048911 |2    |
+----------+-----+
only showing top 20 rows
Rows before deduplication: 102000
Rows after deduplication: 101300
Unique reading IDs: 101300


In [30]:
lazy_pipeline_df = (
    readings_df
    .filter(F.col("reading_status") == "Valid")
    .withColumn(
        "energy_kwh_rounded",
        F.round(F.col("energy_kwh"), 2)
    )
    .groupBy("meter_id")
    .agg(
        F.sum("energy_kwh_rounded").alias("total_energy_kwh")
    )
    .orderBy(F.col("total_energy_kwh").desc())
)

print("Execution plan before running an action:")
lazy_pipeline_df.explain()

print("Running action now:")
lazy_pipeline_df.show(10, truncate=False)

Execution plan before running an action:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_energy_kwh#1264 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_energy_kwh#1264 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2583]
      +- HashAggregate(keys=[meter_id#238], functions=[sum(energy_kwh_rounded#1263)])
         +- Exchange hashpartitioning(meter_id#238, 200), ENSURE_REQUIREMENTS, [plan_id=2580]
            +- HashAggregate(keys=[meter_id#238], functions=[partial_sum(energy_kwh_rounded#1263)])
               +- Project [meter_id#238, round(energy_kwh#240, 2) AS energy_kwh_rounded#1263]
                  +- Filter (isnotnull(reading_status#244) AND (reading_status#244 = Valid))
                     +- FileScan csv [meter_id#238,energy_kwh#240,reading_status#244] Batched: false, DataFilters: [isnotnull(reading_status#244), (reading_status#244 = Valid)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.cs

In [31]:
valid_readings_df = readings_df.filter(
    F.col("reading_status") == "Valid"
)

cached_valid_df = valid_readings_df.cache()

print("Is DataFrame cached?", cached_valid_df.is_cached)

start_time = time.time()
count1 = cached_valid_df.count()
time1 = time.time() - start_time

start_time = time.time()
count2 = cached_valid_df.count()
time2 = time.time() - start_time

print("First count:", count1)
print("First action time:", round(time1, 3), "seconds")

print("Second count:", count2)
print("Second action time:", round(time2, 3), "seconds")

cached_valid_df.unpersist()

print("Cache released.")

Is DataFrame cached? True
First count: 74486
First action time: 2.055 seconds
Second count: 74486
Second action time: 0.16 seconds
Cache released.


In [32]:
outage_readings_df = readings_df.filter(
    F.col("outage_minutes") > 0
).persist(StorageLevel.MEMORY_AND_DISK)

print("Outage reading count:", outage_readings_df.count())

print(
    "Distinct meters with outages:",
    outage_readings_df.select("meter_id").distinct().count()
)

outage_readings_df.unpersist()

print("Persisted data released.")

Outage reading count: 7653
Distinct meters with outages: 150
Persisted data released.


In [33]:
def measure_aggregation(df, label):
    start_time = time.time()

    result_df = (
        df.filter(F.col("reading_status") == "Valid")
        .groupBy("meter_id")
        .agg(F.sum("energy_kwh").alias("total_energy_kwh"))
    )

    result_count = result_df.count()
    elapsed = time.time() - start_time

    print(label)
    print("Result rows:", result_count)
    print("Execution time:", round(elapsed, 3), "seconds")

    return elapsed

uncached_time1 = measure_aggregation(
    readings_df, "Without cache - Run 1"
)

uncached_time2 = measure_aggregation(
    readings_df, "Without cache - Run 2"
)

cached_readings_df = readings_df.cache()
cached_readings_df.count()

cached_time1 = measure_aggregation(
    cached_readings_df, "With cache - Run 1"
)

cached_time2 = measure_aggregation(
    cached_readings_df, "With cache - Run 2"
)

print("\nTiming comparison:")
print("Without cache:", round(uncached_time1, 3), "and", round(uncached_time2, 3), "seconds")
print("With cache:", round(cached_time1, 3), "and", round(cached_time2, 3), "seconds")

cached_readings_df.unpersist()

Without cache - Run 1
Result rows: 150
Execution time: 0.493 seconds
Without cache - Run 2
Result rows: 150
Execution time: 0.465 seconds
With cache - Run 1
Result rows: 150
Execution time: 0.277 seconds
With cache - Run 2
Result rows: 150
Execution time: 0.216 seconds

Timing comparison:
Without cache: 0.493 and 0.465 seconds
With cache: 0.277 and 0.216 seconds


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [34]:
partitioned_df = readings_df.repartition(8)

partition_counts_df = (
    partitioned_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

print("Number of partitions:", partitioned_df.rdd.getNumPartitions())

partition_counts_df.show(8, truncate=False)

Number of partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|0           |11249|
|1           |11249|
|2           |11251|
|3           |11251|
|4           |11250|
|5           |11250|
|6           |11250|
|7           |11250|
+------------+-----+



In [35]:
twelve_partitions_df = readings_df.repartition(12)

print(
    "Initial partition count:",
    twelve_partitions_df.rdd.getNumPartitions()
)

coalesced_df = twelve_partitions_df.coalesce(3)

print(
    "Partitions after coalesce:",
    coalesced_df.rdd.getNumPartitions()
)

print("Execution plan for coalesce(3):")
coalesced_df.explain()

repartitioned_df = twelve_partitions_df.repartition(3)

print(
    "Partitions after repartition:",
    repartitioned_df.rdd.getNumPartitions()
)

print("Execution plan for repartition(3):")
repartitioned_df.explain()

Initial partition count: 12
Partitions after coalesce: 3
Execution plan for coalesce(3):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3577]
            +- FileScan csv [reading_id#237,meter_id#238,reading_timestamp#239,energy_kwh#240,voltage_v#241,power_factor#242,outage_minutes#243,reading_status#244,source_type#245,ambient_temperature_c#246,operator_id#247] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3577]
      +- FileScan csv [reading_id#237,meter_id#238,reading_timestamp#2

In [36]:
cleaned_readings_df = (
    combined_readings_df
    .dropDuplicates(["reading_id"])
    .filter(
        F.col("energy_kwh").isNotNull()
        & F.col("voltage_v").isNotNull()
    )
)

enriched_readings_df = cleaned_readings_df.join(
    meter_master_df,
    on="meter_id",
    how="left"
)

enriched_readings_df = enriched_readings_df.join(
    substation_master_df,
    on="substation_id",
    how="left"
)

print("Enriched readings:")
enriched_readings_df.show(10, truncate=False)

output_path = "/content/enriched_energy_readings_parquet"

(
    enriched_readings_df
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print("Parquet saved at:", output_path)

part_files = sorted(
    f for f in os.listdir(output_path)
    if f.startswith("part-")
)

print("Number of output part files:", len(part_files))
print("Output part files:", part_files)

saved_df = spark.read.parquet(output_path)

print("Saved row count:", saved_df.count())
saved_df.show(10, truncate=False)

Enriched readings:
+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-------------------+-----------------+-----------------+------------+-------------+-------+-----------+-----------------+
|substation_id|meter_id|reading_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|site_type      |tariff_id          |installation_date|rated_capacity_kw|meter_status|city         |region |capacity_mw|commissioned_year|
+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-------------------+-----------------+-----------------+------------+-------------+-------+-----------+-----------------+
|SUB05        |M0020   |RD0000002 |2026-01-31 06:00:00|2.97      |227.4 

**Final Capstone**

In [37]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

combined_readings_df = readings_df.unionByName(
    readings_extra_df,
    allowMissingColumns=True
)

print("Combined rows:", combined_readings_df.count())

capstone_readings_df = combined_readings_df.dropDuplicates(
    ["reading_id"]
)

print("Rows after deduplication:", capstone_readings_df.count())

Combined rows: 102000
Rows after deduplication: 101300


In [38]:
cleaned_readings_df = (
    capstone_readings_df
    .withColumn(
        "reading_timestamp",
        F.to_timestamp("reading_timestamp")
    )
    .filter(F.col("energy_kwh").isNotNull())
)

billable_readings_df = cleaned_readings_df.filter(
    F.col("reading_status") != "Rejected"
)

print("Cleaned rows:", cleaned_readings_df.count())
print("Non-Rejected rows:", billable_readings_df.count())

Cleaned rows: 99477
Non-Rejected rows: 95408


In [39]:
meter_details_df = meter_master_df.join(
    substation_master_df,
    on="substation_id",
    how="left"
)

capstone_enriched_df = (
    cleaned_readings_df
    .join(meter_details_df, on="meter_id", how="left")
    .join(tariff_master_df, on="tariff_id", how="left")
)

print("Enriched data:")
capstone_enriched_df.show(10, truncate=False)

Enriched data:
+-------------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-------------+---------------+-----------------+-----------------+------------+-------------+-------+-----------+-----------------+--------------------+---------------+----------------+------------------------+
|tariff_id          |meter_id|reading_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|substation_id|site_type      |installation_date|rated_capacity_kw|meter_status|city         |region |capacity_mw|commissioned_year|tariff_name         |site_category  |rate_per_kwh_inr|monthly_fixed_charge_inr|
+-------------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-------------+---------------+-----------------+--

In [40]:
capstone_enriched_df = (
    capstone_enriched_df
    .withColumn(
        "month",
        F.date_format("reading_timestamp", "yyyy-MM")
    )
    .withColumn(
        "billable_energy_kwh",
        F.when(
            (F.col("reading_status") != "Rejected") &
            F.col("energy_kwh").isNotNull(),
            F.col("energy_kwh")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "energy_charge_inr",
        F.when(
            (F.col("reading_status") != "Rejected") &
            F.col("energy_kwh").isNotNull() &
            F.col("rate_per_kwh_inr").isNotNull(),
            F.col("energy_kwh") * F.col("rate_per_kwh_inr")
        ).otherwise(F.lit(0.0))
    )
)

capstone_enriched_df.select(
    "month",
    "reading_id",
    "region",
    "site_type",
    "reading_status",
    "billable_energy_kwh",
    "energy_charge_inr"
).show(10, truncate=False)

+-------+----------+-------+---------------+--------------+-------------------+------------------+
|month  |reading_id|region |site_type      |reading_status|billable_energy_kwh|energy_charge_inr |
+-------+----------+-------+---------------+--------------+-------------------+------------------+
|2026-01|RD0000002 |West   |Residential    |Valid         |2.97               |19.008000000000003|
|2026-04|RD0000003 |Central|Commercial     |Estimated     |8.61               |70.60199999999999 |
|2026-03|RD0000004 |Central|Residential    |Valid         |1.87               |9.536999999999999 |
|2026-05|RD0000005 |Central|Commercial     |Valid         |4.39               |35.998            |
|2026-02|RD0000011 |South  |Public Facility|Valid         |10.51              |70.417            |
|2026-05|RD0000012 |South  |Residential    |Estimated     |1.49               |9.536             |
|2026-04|RD0000016 |South  |Public Facility|Valid         |25.82              |172.994           |
|2026-06|R

In [41]:
capstone_report_df = (
    capstone_enriched_df
    .withColumn(
        "reports_outage",
        F.when(
            F.coalesce(F.col("outage_minutes"), F.lit(0)) > 0,
            1
        ).otherwise(0)
    )
    .groupBy("month", "region", "site_type")
    .agg(
        F.sum("billable_energy_kwh").alias("total_billable_kwh"),
        F.sum("energy_charge_inr").alias("estimated_energy_charges_inr"),
        F.sum(
            F.coalesce(F.col("outage_minutes"), F.lit(0))
        ).alias("total_outage_minutes"),
        F.count("*").alias("total_readings"),
        F.sum("reports_outage").alias("readings_reporting_outage")
    )
    .withColumn(
        "outage_reading_percentage",
        F.when(
            F.col("total_readings") > 0,
            F.col("readings_reporting_outage") * 100.0 /
            F.col("total_readings")
        ).otherwise(F.lit(0.0))
    )
)

capstone_report_df.show(100, truncate=False)

+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+
|month  |region |site_type      |total_billable_kwh|estimated_energy_charges_inr|total_outage_minutes|total_readings|readings_reporting_outage|outage_reading_percentage|
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+
|2026-04|West   |Commercial     |20686.28          |169627.4960000001           |8433                |1138          |96                       |8.43585237258348         |
|2026-02|South  |Residential    |8840.33           |51504.692999999985          |23312               |2528          |241                      |9.533227848101266        |
|2026-08|West   |Residential    |2520.0            |14947.223000000005          |6589                |730           |66                       |9.04109

In [42]:
charge_rank_window = Window.partitionBy(
    "month", "region"
).orderBy(
    F.col("estimated_energy_charges_inr").desc_nulls_last()
)

capstone_report_df = (
    capstone_report_df
    .withColumn(
        "site_type_charge_rank",
        F.dense_rank().over(charge_rank_window)
    )
    .orderBy("month", "region", "site_type_charge_rank")
)

capstone_report_df.show(100, truncate=False)

+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|month  |region |site_type      |total_billable_kwh|estimated_energy_charges_inr|total_outage_minutes|total_readings|readings_reporting_outage|outage_reading_percentage|site_type_charge_rank|
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|2026-01|Central|Industrial     |24871.22          |226328.102                  |4396                |432           |49                       |11.342592592592593       |1                    |
|2026-01|Central|Commercial     |10649.029999999995|87322.04599999999           |5305                |586           |60                       |10.238907849829351       |2                    |
|2026-01|Central|Residential    |814.03 

In [43]:
charge_rank_window = Window.partitionBy(
    "month", "region"
).orderBy(
    F.col("estimated_energy_charges_inr").desc_nulls_last()
)

capstone_report_df = (
    capstone_report_df
    .withColumn(
        "site_type_charge_rank",
        F.dense_rank().over(charge_rank_window)
    )
    .orderBy("month", "region", "site_type_charge_rank")
)

capstone_report_df.show(100, truncate=False)

+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|month  |region |site_type      |total_billable_kwh|estimated_energy_charges_inr|total_outage_minutes|total_readings|readings_reporting_outage|outage_reading_percentage|site_type_charge_rank|
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|2026-01|Central|Industrial     |24871.22          |226328.102                  |4396                |432           |49                       |11.342592592592593       |1                    |
|2026-01|Central|Commercial     |10649.029999999995|87322.04599999999           |5305                |586           |60                       |10.238907849829351       |2                    |
|2026-01|Central|Residential    |814.03 

In [44]:
capstone_cached_df = capstone_report_df.cache()

print("Report row count:", capstone_cached_df.count())

print(
    "Distinct regions:",
    capstone_cached_df.select("region").distinct().count()
)

capstone_cached_df.unpersist()

print("Cached report released.")

Report row count: 120
Distinct regions: 4
Cached report released.


In [45]:
import os

capstone_output_path = "/content/energy_monthly_reliability_report_parquet"

(
    capstone_report_df
    .repartition(2)
    .write
    .mode("overwrite")
    .parquet(capstone_output_path)
)

print("Report saved to:", capstone_output_path)

part_files = sorted(
    f for f in os.listdir(capstone_output_path)
    if f.startswith("part-")
)

print("Number of part files:", len(part_files))
print("Part files:", part_files)

saved_report_df = spark.read.parquet(capstone_output_path)

print("Saved report rows:", saved_report_df.count())
saved_report_df.show(20, truncate=False)

Report saved to: /content/energy_monthly_reliability_report_parquet
Number of part files: 2
Part files: ['part-00000-ef29941d-4024-40b0-b5a9-f0b5b68b9fd0-c000.snappy.parquet', 'part-00001-ef29941d-4024-40b0-b5a9-f0b5b68b9fd0-c000.snappy.parquet']
Saved report rows: 120
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|month  |region |site_type      |total_billable_kwh|estimated_energy_charges_inr|total_outage_minutes|total_readings|readings_reporting_outage|outage_reading_percentage|site_type_charge_rank|
+-------+-------+---------------+------------------+----------------------------+--------------------+--------------+-------------------------+-------------------------+---------------------+
|2026-05|South  |Residential    |10061.610000000004|58778.71999999996           |21657               |2847          |230                      |8.078679311